# 🔮 Notebook 6 — Prediction Demo

**Project:** Customer Churn Prediction  
**Objective:** Demonstrate how to use the saved model to predict churn for individual customers.

This notebook is the notebook equivalent of `src/predict.py`.
It shows the full prediction pipeline for 3 sample customer profiles:
- **Profile A:** High-risk (month-to-month, fiber optic, short tenure)
- **Profile B:** Low-risk (two-year contract, long tenure)
- **Profile C:** Medium-risk edge case

---

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from utils import load_model
from predict import preprocess_input, get_retention_actions, get_top_reasons

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

MODEL_PATH = os.path.join('..', 'model', 'churn_model.pkl')
print('Libraries loaded ✅')

## 6.1  Load the Saved Model

In [ ]:
bundle        = load_model(MODEL_PATH)
model         = bundle['model']
model_name    = bundle['model_name']
scaler        = bundle['scaler']
feature_names = bundle['feature_names']

print(f'Model      : {model_name}')
print(f'Features   : {len(feature_names)}')
print(f'Scaler     : {type(scaler).__name__}')

## 6.2  Define Sample Customer Profiles

In [ ]:
# ── Profile A: HIGH RISK ─────────────────────────────────────────────────
# Month-to-month contract, fiber optic, short tenure, no add-ons, electronic check
profile_A = {
    'SeniorCitizen'   : 1,
    'gender'          : 'Female',
    'Partner'         : 'No',
    'Dependents'      : 'No',
    'tenure'          : 3,
    'PhoneService'    : 'Yes',
    'MultipleLines'   : 'No',
    'InternetService' : 'Fiber optic',
    'OnlineSecurity'  : 'No',
    'OnlineBackup'    : 'No',
    'DeviceProtection': 'No',
    'TechSupport'     : 'No',
    'StreamingTV'     : 'Yes',
    'StreamingMovies' : 'Yes',
    'Contract'        : 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod'   : 'Electronic check',
    'MonthlyCharges'  : 95.00,
    'TotalCharges'    : 285.00,
}

# ── Profile B: LOW RISK ──────────────────────────────────────────────────
# Two-year contract, DSL, long tenure, all add-ons, automatic payment
profile_B = {
    'SeniorCitizen'   : 0,
    'gender'          : 'Male',
    'Partner'         : 'Yes',
    'Dependents'      : 'Yes',
    'tenure'          : 58,
    'PhoneService'    : 'Yes',
    'MultipleLines'   : 'Yes',
    'InternetService' : 'DSL',
    'OnlineSecurity'  : 'Yes',
    'OnlineBackup'    : 'Yes',
    'DeviceProtection': 'Yes',
    'TechSupport'     : 'Yes',
    'StreamingTV'     : 'No',
    'StreamingMovies' : 'No',
    'Contract'        : 'Two year',
    'PaperlessBilling': 'No',
    'PaymentMethod'   : 'Bank transfer (automatic)',
    'MonthlyCharges'  : 55.50,
    'TotalCharges'    : 3219.00,
}

# ── Profile C: MEDIUM RISK ───────────────────────────────────────────────
# One-year contract, fiber optic, medium tenure, partial add-ons
profile_C = {
    'SeniorCitizen'   : 0,
    'gender'          : 'Female',
    'Partner'         : 'No',
    'Dependents'      : 'No',
    'tenure'          : 24,
    'PhoneService'    : 'Yes',
    'MultipleLines'   : 'No',
    'InternetService' : 'Fiber optic',
    'OnlineSecurity'  : 'Yes',
    'OnlineBackup'    : 'No',
    'DeviceProtection': 'Yes',
    'TechSupport'     : 'No',
    'StreamingTV'     : 'Yes',
    'StreamingMovies' : 'No',
    'Contract'        : 'One year',
    'PaperlessBilling': 'Yes',
    'PaymentMethod'   : 'Credit card (automatic)',
    'MonthlyCharges'  : 74.25,
    'TotalCharges'    : 1782.00,
}

profiles = [
    ('Profile A — HIGH RISK',   profile_A),
    ('Profile B — LOW RISK',    profile_B),
    ('Profile C — MEDIUM RISK', profile_C),
]

print('Sample profiles defined ✅')

## 6.3  Run Predictions

In [ ]:
def predict_customer(raw: dict, model, scaler, feature_names: list) -> tuple:
    """Preprocess raw input and return (label, probability)."""
    X = preprocess_input(raw, feature_names, scaler)
    label = model.predict(X)[0]
    prob  = model.predict_proba(X)[0][1] if hasattr(model, 'predict_proba') else float(label)
    return ('Yes' if label == 1 else 'No'), prob, X


results = []
for title, profile in profiles:
    label, prob, X_input = predict_customer(profile, model, scaler, feature_names)
    results.append((title, profile, label, prob, X_input))

print('Predictions complete ✅')

## 6.4  Display Prediction Results

In [ ]:
for title, profile, label, prob, X_input in results:
    if prob >= 0.75:
        risk_icon = '🔴'
    elif prob >= 0.45:
        risk_icon = '🟡'
    else:
        risk_icon = '🟢'

    print('\n' + '=' * 60)
    print(f'  {title}')
    print('=' * 60)
    print(f'  Predicted Churn    : {label}')
    print(f'  Probability        : {prob * 100:.1f}%')
    print(f'  Risk Level         : {risk_icon}')

    print('\n  TOP INFLUENCING FACTORS:')
    print('-' * 60)
    reasons = get_top_reasons(model, X_input, feature_names, top_n=5)
    for r in reasons:
        print(r)

    print('\n  RETENTION ACTIONS:')
    print('-' * 60)
    for action in get_retention_actions(profile, prob):
        print(f'  {action}')
    print('=' * 60)

## 6.5  Churn Probability Visualisation

In [ ]:
profile_names = [r[0].split('—')[0].strip() for r in results]
probs         = [r[3] * 100 for r in results]
colors        = ['#F44336' if p >= 75 else '#FF9800' if p >= 45 else '#4CAF50' for p in probs]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(profile_names, probs, color=colors, edgecolor='white', height=0.5)

for bar, prob in zip(bars, probs):
    ax.text(
        bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
        f'{prob:.1f}%', va='center', fontsize=12, fontweight='bold'
    )

ax.axvline(50, color='black', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_xlim(0, 110)
ax.set_xlabel('Churn Probability (%)', fontsize=12)
ax.set_title('Predicted Churn Probability — Sample Customers', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()

PLOTS_DIR = os.path.join('..', 'outputs', 'plots')
plt.savefig(os.path.join(PLOTS_DIR, '12_prediction_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.6  Predict a Custom Customer (Interactive)

In [ ]:
# Modify this dict and re-run the cell to test any customer
custom_customer = {
    'SeniorCitizen'   : 0,
    'gender'          : 'Male',
    'Partner'         : 'No',
    'Dependents'      : 'No',
    'tenure'          : 10,
    'PhoneService'    : 'Yes',
    'MultipleLines'   : 'No',
    'InternetService' : 'Fiber optic',
    'OnlineSecurity'  : 'No',
    'OnlineBackup'    : 'No',
    'DeviceProtection': 'No',
    'TechSupport'     : 'No',
    'StreamingTV'     : 'No',
    'StreamingMovies' : 'No',
    'Contract'        : 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod'   : 'Electronic check',
    'MonthlyCharges'  : 70.00,
    'TotalCharges'    : 700.00,
}

label, prob, X_input = predict_customer(custom_customer, model, scaler, feature_names)

print(f'Predicted Churn   : {label}')
print(f'Churn Probability : {prob * 100:.1f}%')
print('\nTop Influencing Factors:')
for r in get_top_reasons(model, X_input, feature_names, top_n=5):
    print(r)
print('\nSuggested Retention Actions:')
for a in get_retention_actions(custom_customer, prob):
    print(f'  {a}')

---
## Summary

This notebook demonstrated the complete end-to-end prediction pipeline:

1. Load saved model bundle
2. Preprocess raw customer data (same pipeline as training)
3. Predict churn probability
4. Explain prediction with top features / SHAP
5. Suggest personalised retention actions

For terminal-based interactive prediction, run:
```bash
python src/predict.py
```

---
**Project complete ✅**